# Notebook 01 — Exploratory Data Analysis

Loads the cleaned Delhivery dataset and explores key distributions:
- Delay ratio distribution (actual / OSRM)
- Route type breakdown (FTL vs Carting)
- Time-of-day patterns
- Distance vs. delay scatter
- City-level delay overview
- Chronic corridor identification

In [1]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from src.data.pipeline import run as run_pipeline

plt.style.use('dark_background')
sns.set_palette('husl')
pd.options.display.max_columns = 50
print('Libraries loaded')

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
# Load or run pipeline
import os
if not os.path.exists('data/processed/delhivery_clean.parquet'):
    print('Running pipeline...')
    df = run_pipeline()
else:
    df = pd.read_parquet('data/processed/delhivery_clean.parquet')

print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Basic stats
print('=== DATA TYPES ===')
print(df.dtypes)
print('\n=== MISSING VALUES ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== NUMERIC SUMMARY ===')
df.describe()

In [2]:
# Delay ratio distribution
if 'delay_ratio_raw' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df['delay_ratio_raw'].hist(bins=80, ax=axes[0], color='#4f90ff', edgecolor='none')
    axes[0].axvline(1.0, color='green', lw=2, label='On-time (ratio=1.0)')
    axes[0].axvline(1.2, color='red', lw=2, linestyle='--', label='Chronic threshold (1.2)')
    axes[0].set_title('Delay Ratio Distribution (actual / OSRM)')
    axes[0].legend()

    if 'route_type' in df.columns:
        for rt, grp in df.groupby('route_type'):
            grp['delay_ratio_raw'].hist(bins=60, ax=axes[1], alpha=0.6, label=rt)
        axes[1].set_title('Delay Ratio by Route Type (FTL vs Carting)')
        axes[1].legend()
    plt.tight_layout()
    plt.savefig('reports/01_delay_ratio_dist.png', dpi=150, bbox_inches='tight')
    plt.show()

NameError: name 'df' is not defined

In [ ]:
# Time of day analysis
if 'time_of_day' in df.columns and 'delay_ratio_raw' in df.columns:
    tod_stats = df.groupby('time_of_day')['delay_ratio_raw'].agg(['mean','median','std']).reset_index()
    print('Time-of-day delay stats:')
    print(tod_stats)
    
    fig = px.box(df, x='time_of_day', y='delay_ratio_raw', color='route_type',
                 category_orders={'time_of_day': ['morning','afternoon','evening','night']},
                 title='Delay Ratio by Time of Day and Route Type',
                 template='plotly_dark')
    fig.add_hline(y=1.2, line_dash='dash', line_color='red', annotation_text='Chronic threshold')
    fig.show()

In [ ]:
# Distance vs Delay
dist_col = next((c for c in df.columns if 'distance' in c and 'osrm' in c), None)
if dist_col and 'delay_ratio_raw' in df.columns:
    sample = df.sample(min(5000, len(df)), random_state=42)
    fig = px.scatter(sample, x=dist_col, y='delay_ratio_raw',
                     color='route_type', opacity=0.4,
                     title='Distance vs Delay Ratio',
                     template='plotly_dark',
                     trendline='ols')
    fig.add_hline(y=1.2, line_dash='dash', line_color='red')
    fig.show()

In [ ]:
# Chronic corridor summary
if 'is_chronic_corridor' in df.columns:
    chronic_pct = df['is_chronic_corridor'].mean() * 100
    print(f'Chronic corridors: {chronic_pct:.1f}% of trips')
    
    if 'route_type' in df.columns:
        print(df.groupby('route_type')['is_chronic_corridor'].mean().rename('chronic_rate').to_frame())
    
print('\n=== Top 10 Worst Corridors ===')
if 'corridor_median_delay' in df.columns and 'source_name' in df.columns:
    top_corridors = (
        df[['source_name','destination_name','corridor_median_delay','corridor_trip_count']]
        .drop_duplicates()
        .sort_values('corridor_median_delay', ascending=False)
        .head(10)
    )
    print(top_corridors.to_string())